### 2 - Teste Cálculos Indicadores

In [20]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_parquet("../dados_tratados/teste_restinga_dados_preparados.parquet")

print("Dimensão original df:", df.shape)

Dimensão original df: (10445, 24)


In [21]:
df.head()

,Ano,Carga Horaria,Carga Horaria Mínima,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Fator Esforço Curso,Idade,Modalidade de Ensino,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Tipo de Oferta,Turno
0,2017,2090,2000.0,Em curso,Não declarada,44681986,1207788,2014-12-22,2012-03-05,2012-03-01,Informação e Comunicação,35 a 39 anos,"1,25",35.0,Educação Presencial,2018-03-05,Análise e Desenvolvimento de Sistemas,Não declarada,F,Em curso,Informática,Tecnologia,Não se aplica,Matutino
1,2017,3893,1200.0,Em curso,Branca,66262145,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,"1,27",18.0,Educação Presencial,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em curso,Elétrica,Técnico,Integrado,Matutino
2,2017,3893,1200.0,Em curso,Branca,66262155,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,"1,27",17.0,Educação Presencial,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em curso,Elétrica,Técnico,Integrado,Matutino
3,2017,3893,1200.0,Em curso,Branca,66262151,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,"1,27",18.0,Educação Presencial,2016-02-01,Técnico em Eletrônica,"0<RFP<=0,5",F,Em curso,Elétrica,Técnico,Integrado,Matutino
4,2017,3893,1200.0,Evadidos,Branca,66262139,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,"1,27",17.0,Educação Presencial,2017-05-01,Técnico em Eletrônica,"0<RFP<=0,5",F,Transf. externa,Elétrica,Técnico,Integrado,Matutino


In [22]:
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10445 entries, 0 to 10444
Data columns (total 24 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Ano                              10445 non-null  int64         
 1   Carga Horaria                    10445 non-null  int64         
 2   Carga Horaria Mínima             10445 non-null  float64       
 3   Categoria da Situação            10445 non-null  object        
 4   Cor / Raça                       10445 non-null  object        
 5   Código da Matricula              10445 non-null  int64         
 6   Código do Ciclo Matricula        10445 non-null  int64         
 7   Data de Fim Previsto do Ciclo    10445 non-null  datetime64[ns]
 8   Data de Inicio do Ciclo          10445 non-null  datetime64[ns]
 9   Data de Ocorrencia da Matricula  10445 non-null  datetime64[ns]
 10  Eixo Tecnológico                 10445 non-null  object   

Testando alguns agrupamentos:

In [23]:
contagem_matriculas_ano_curso = df.groupby(["Ano", "Nome de Curso"])["Código da Matricula"].count().reset_index() #resetamos o index para permitir agrupar/salvar em excel/csv
print(contagem_matriculas_ano_curso)

     Ano                          Nome de Curso  Código da Matricula
0   2017  Análise e Desenvolvimento de Sistemas                  247
1   2017                  Eletrônica Industrial                   84
2   2017           Gestão Desportiva e de Lazer                  112
3   2017          Letras - Português e Espanhol                   30
4   2017                    Técnico em Comércio                   33
..   ...                                    ...                  ...
81  2024                    Técnico em Comércio                  127
82  2024                  Técnico em Eletrônica                  164
83  2024             Técnico em Guia de Turismo                   77
84  2024                 Técnico em Informática                  217
85  2024                       Técnico em Lazer                  117

[86 rows x 3 columns]


In [ ]:
#df.to_csv("teste_restinga_apos_conversao_datas.csv", index=False)
#df.describe()


#### Ajustes dos dados das situações de matrícula:

In [24]:
df["Situação de Matrícula"].unique()

array(['Em curso', 'Transf. externa', 'Desligada', 'Abandono',
       'Concluída', 'Integralizada', 'Transf. interna'], dtype=object)

In [26]:
periodo_fim_previsto = df["Data de Fim Previsto do Ciclo"].dt.to_period("M")
periodo_referencia = pd.PeriodIndex.from_fields(
    year=df["Ano"],
    month=12,
    freq="M"
)

df.loc[df["Situação de Matrícula"] == "Em curso", "Situação de Matrícula"] = np.where(
    periodo_fim_previsto[df["Situação de Matrícula"] == "Em curso"] < periodo_referencia[df["Situação de Matrícula"] == "Em curso"],
    "Retido",
    "Em fluxo"
)

df["Situação de Matrícula"].value_counts()

Situação de Matrícula
Em fluxo           6684
Retido             2316
Concluída           638
Abandono            449
Desligada           244
Transf. externa     103
Transf. interna       6
Integralizada         5
Name: count, dtype: int64

In [27]:
df.to_csv("teste_restinga_retidos_emcurso3.csv", index=False)

#### Indicadores por Curso e Ano:

In [13]:
indicadores_curso_ano = (
    df.groupby(["Ano", "Nome de Curso"])
      .agg(
          matriculas_atendidas = ("Código da Matricula", "nunique"),
          matriculas_ativas_regulares = ("Situação de Matrícula", lambda x: (x == "Em fluxo").sum()),
          matriculas_ativas_retidas = ("Situação de Matrícula", lambda x: (x == "Retido").sum()),
          concluintes= ("Situação de Matrícula", lambda x: (x == "Concluída").sum()),
          abandono = ("Situação de Matrícula", lambda x: (x == "Abandono").sum()),
          desligado = ("Situação de Matrícula", lambda x: (x == "Desligada").sum()),
          transferencia_interna = ("Situação de Matrícula", lambda x: (x == "Transf. externa").sum()),
          transferencia_externa = ("Situação de Matrícula", lambda x: (x == "Transf. interna").sum()),
          integralizada = ("Situação de Matrícula", lambda x: (x == "Integralizada").sum()),
        )
      .reset_index()
)

indicadores_curso_ano

,Ano,Nome de Curso,matriculas_atendidas,matriculas_ativas_regulares,matriculas_ativas_retidas,concluintes,abandono,desligado,transferencia_interna,transferencia_externa,integralizada
0,2017,Análise e Desenvolvimento de Sistemas,247,163,31,17,25,8,2,1,0
1,2017,Eletrônica Industrial,84,54,5,0,19,6,0,0,0
2,2017,Gestão Desportiva e de Lazer,112,74,6,12,13,5,0,0,2
3,2017,Letras - Português e Espanhol,30,25,0,0,0,5,0,0,0
4,2017,Técnico em Comércio,33,33,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
81,2024,Técnico em Comércio,127,84,35,3,0,5,0,0,0
82,2024,Técnico em Eletrônica,164,95,59,6,0,3,1,0,0
83,2024,Técnico em Guia de Turismo,77,37,32,8,0,0,0,0,0
84,2024,Técnico em Informática,217,138,51,21,0,4,3,0,0


In [ ]:
def calcular_indicadores(df, agrupamento):
    indicadores = (
        df.groupby(agrupamento)
          .agg(
            matriculas_atendidas = ("Código da Matricula", "nunique"),
            matriculas_ativas_regulares = ("Situação de Matrícula", lambda x: (x == "Em fluxo").sum()),
            matriculas_ativas_retidas = ("Situação de Matrícula", lambda x: (x == "Retido").sum()),
            concluintes = ("Situação de Matrícula", lambda x: (x == "Concluída").sum()),
            abandono = ("Situação de Matrícula", lambda x: (x == "Abandono").sum()),
            desligados = ("Situação de Matrícula", lambda x: (x == "Desligada").sum()),
            transferencia_interna = ("Situação de Matrícula", lambda x: (x == "Transf. externa").sum()),
            transferencia_externa = ("Situação de Matrícula", lambda x: (x == "Transf. interna").sum()),
            integralizadas = ("Situação de Matrícula", lambda x: (x == "Integralizada").sum()),
          )
          .reset_index()
    )

    indicadores["taxa_conclusao"] = (
        (indicadores["concluintes"] + indicadores["integralizadas"]) / indicadores["matriculas_atendidas"] * 100
    )
    indicadores["taxa_evasao"] = (
        (indicadores["abandono"] + indicadores["desligados"] + indicadores["transferencia_externa"]) / indicadores["matriculas_atendidas"] * 100
    )
    indicadores["taxa_retencao"] = (
        indicadores["matriculas_ativas_retidas"] / indicadores["matriculas_atendidas"] * 100
    )
    #indicadores["indice_eficiencia"] = 
    
    
    

    return indicadores.round(2)